[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/IntroduccionAstronomiaPractica/blob/main/CirculosPrincipales-CoordenadasCelestes/Analisis.ipynb)

In [ ]:
!pip install astroquery -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.2 MB/s eta 0:00:00


In [18]:
"""
Sun Ephemeris Query — JPL Horizons via astroquery
==================================================
Target  : Sol (Sun) [ID=10]
Observer: 6.2675°N, 75.2675°W, 1495 m  (lon/lat/elev)
Period  : 2026-04-03 00:00 UT -> 2026-04-04 00:00 UT
Step    : 1 minute  (1441 epochs)

Columns retrieved
-----------------
  datetime_str   — epoch label from Horizons
  datetime_jd    — Julian Date
  RA             — Astrometric Right Ascension  [deg, J2000/ICRF]
  DEC            — Astrometric Declination       [deg, J2000/ICRF]
  AZ             — Apparent Azimuth              [deg, N->E convention]
  EL             — Apparent Elevation            [deg]

Quantities used (Horizons codes)
---------------------------------
  1  -> Astrometric RA & DEC (J2000/ICRF)
  4  -> Apparent AZ & EL (airless, i.e. no refraction correction)

Install requirements
--------------------
  pip install astroquery pandas
"""

import pandas as pd
from astroquery.jplhorizons import Horizons

# ── Observer location ────────────────────────────────────────────────────────
# Format expected by Horizons: {'lon': deg_E, 'lat': deg_N, 'elevation': km}
# Note: negative latitude → Southern Hemisphere
LOCATION = {
    "lon": -75.267452683656837,
    "lat": 6.267452683656837,
    "elevation": 1.495,               # km above WGS-84 ellipsoid
}

# ── Epochs ───────────────────────────────────────────────────────────────────
EPOCHS = {
    "start": "2026-04-03 00:00",
    "stop":  "2026-04-04 00:00",
    "step":  "10m",               # 1-minute cadence
}

# ── Query ────────────────────────────────────────────────────────────────────
print("Querying JPL Horizons …")
obj = Horizons(
    id="10",           # Sun
    location=LOCATION,
    epochs=EPOCHS,
)

eph = obj.ephemerides(
    quantities="1,4",
    skip_daylight=False,
)

# ── Build DataFrame ──────────────────────────────────────────────────────────
# Convert AstroPy table -> pandas, then keep only the columns we care about
df_raw = eph.to_pandas()

df = df_raw[["datetime_str", "datetime_jd", "RA", "DEC", "AZ", "EL"]].copy()
df["Epoch (UT)"] = pd.to_datetime(df_raw["datetime_str"], format="%Y-%b-%d %H:%M")
df["Epoch (COL)"] = df["Epoch (UT)"].dt.tz_localize("UTC").dt.tz_convert("America/Bogota")

# ── Inspect & save ───────────────────────────────────────────────────────────
print(f"\nShape  : {df.shape[0]} rows × {df.shape[1]} columns")
display(df)

Querying JPL Horizons …

Shape  : 145 rows × 8 columns


,datetime_str,datetime_jd,RA,DEC,AZ,EL,Epoch (UT),Epoch (COL)
0,2026-Apr-03 00:00,2.461134e+06,11.85194,5.08784,276.883137,-13.158144,2026-04-03 00:00:00,2026-04-02 19:00:00-05:00
1,2026-Apr-03 00:10,2.461134e+06,11.85830,5.09049,277.237273,-15.624481,2026-04-03 00:10:00,2026-04-02 19:10:00-05:00
2,2026-Apr-03 00:20,2.461134e+06,11.86466,5.09315,277.610359,-18.088811,2026-04-03 00:20:00,2026-04-02 19:20:00-05:00
3,2026-Apr-03 00:30,2.461134e+06,11.87103,5.09580,278.004549,-20.550917,2026-04-03 00:30:00,2026-04-02 19:30:00-05:00
4,2026-Apr-03 00:40,2.461134e+06,11.87740,5.09846,278.422298,-23.010548,2026-04-03 00:40:00,2026-04-02 19:40:00-05:00
...,...,...,...,...,...,...,...,...
140,2026-Apr-03 23:20,2.461134e+06,12.73841,5.46015,276.010043,-3.304666,2026-04-03 23:20:00,2026-04-03 18:20:00-05:00
141,2026-Apr-03 23:30,2.461134e+06,12.74475,5.46280,276.306872,-5.775568,2026-04-03 23:30:00,2026-04-03 18:30:00-05:00
142,2026-Apr-03 23:40,2.461134e+06,12.75110,5.46544,276.616936,-8.245008,2026-04-03 23:40:00,2026-04-03 18:40:00-05:00
143,2026-Apr-03 23:50,2.461134e+06,12.75745,5.46808,276.941608,-10.712843,2026-04-03 23:50:00,2026-04-03 18:50:00-05:00


In [19]:
df_filtrado = df.set_index("Epoch (COL)").between_time("15:00", "16:00").reset_index()
df_filtrado

,Epoch (COL),datetime_str,datetime_jd,RA,DEC,AZ,EL,Epoch (UT)
0,2026-04-03 15:00:00-05:00,2026-Apr-03 20:00,2.461134e+06,12.61246,5.40725,271.483116,46.291889,2026-04-03 20:00:00
1,2026-04-03 15:10:00-05:00,2026-Apr-03 20:10,2.461134e+06,12.61872,5.40989,271.690916,43.807357,2026-04-03 20:10:00
2,2026-04-03 15:20:00-05:00,2026-Apr-03 20:20,2.461134e+06,12.62499,5.41254,271.896073,41.323098,2026-04-03 20:20:00
3,2026-04-03 15:30:00-05:00,2026-Apr-03 20:30,2.461134e+06,12.63125,5.41519,272.099621,38.839139,2026-04-03 20:30:00
4,2026-04-03 15:40:00-05:00,2026-Apr-03 20:40,2.461134e+06,12.63752,5.41784,272.302461,36.355513,2026-04-03 20:40:00
5,2026-04-03 15:50:00-05:00,2026-Apr-03 20:50,2.461134e+06,12.64379,5.42048,272.505394,33.872249,2026-04-03 20:50:00
6,2026-04-03 16:00:00-05:00,2026-Apr-03 21:00,2.461134e+06,12.65007,5.42313,272.709146,31.389381,2026-04-03 21:00:00


## Análisis por fechas específicas

Se obtienen los datos de **RA, DEC, AZ y EL** para el Sol en las siguientes
fechas y horas (hora Colombia, **UT−5**):

| # | Fecha y hora (COL) |
|---|-------------------|
| 1 | 2026-04-03 15:38  |
| 2 | 2026-04-04 16:02  |
| 3 | 2026-04-06 16:50  |
| 4 | 2026-04-08 17:05  |

Para cada fecha se consulta una ventana de **±30 minutos** alrededor de la hora
indicada, con un paso de **1 minuto**. Se muestra explícitamente la coordenada
correspondiente a cada hora especificada y se marca con un **punto rojo** en los
gráficos RA–DEC y AZ–EL.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from astroquery.jplhorizons import Horizons

%matplotlib inline

# ── Observer location (same as above) ────────────────────────────────────────
LOCATION = {
    "lon": -75.267452683656837,
    "lat":  6.267452683656837,
    "elevation": 1.495,   # km
}

# ── Observation times in Colombia time (UT-5) ─────────────────────────────────
OBSERVATIONS = [
    "2026-04-03 15:38",
    "2026-04-04 16:02",
    "2026-04-06 16:50",
    "2026-04-08 17:05",
]

In [ ]:
def query_sun_window(t_col_str, delta_min=30, step="1m"):
    """Query Sun ephemeris ±delta_min around t_col_str (Colombia / UT-5)."""
    t_col   = pd.Timestamp(t_col_str, tz="America/Bogota")
    t_utc   = t_col.tz_convert("UTC")
    t_start = t_utc - pd.Timedelta(minutes=delta_min)
    t_stop  = t_utc + pd.Timedelta(minutes=delta_min)

    epochs = {
        "start": t_start.strftime("%Y-%m-%d %H:%M"),
        "stop":  t_stop.strftime("%Y-%m-%d %H:%M"),
        "step":  step,
    }

    print(f"  Consultando Horizons: {epochs['start']} – {epochs['stop']} UT "
          f"(paso {step}) …")
    obj = Horizons(id="10", location=LOCATION, epochs=epochs)
    eph = obj.ephemerides(quantities="1,4", skip_daylight=False)

    df = eph.to_pandas()[["datetime_str", "datetime_jd", "RA", "DEC", "AZ", "EL"]].copy()
    df["Epoch (UT)"]  = pd.to_datetime(df["datetime_str"], format="%Y-%b-%d %H:%M")
    df["Epoch (COL)"] = (
        df["Epoch (UT)"]
        .dt.tz_localize("UTC")
        .dt.tz_convert("America/Bogota")
    )
    return df, t_col


for t_col_str in OBSERVATIONS:
    print(f"\n{'='*65}")
    print(f"  Observación: {t_col_str} COL (UT-5)")
    print(f"{'='*65}")

    df_obs, t_col = query_sun_window(t_col_str)

    # ── Find epoch closest to the target time ────────────────────────────────
    time_diff = (df_obs["Epoch (COL)"] - t_col).abs()
    idx_ref   = time_diff.idxmin()
    row       = df_obs.loc[idx_ref]

    # ── Print coordinates at target time ─────────────────────────────────────
    print(f"\n  Coordenadas en {t_col_str} COL:")
    print(f"    Epoch (UT)  : {row['Epoch (UT)']}")
    print(f"    Epoch (COL) : {row['Epoch (COL)']}")
    print(f"    RA          : {row['RA']:.4f} °")
    print(f"    DEC         : {row['DEC']:.4f} °")
    print(f"    AZ          : {row['AZ']:.4f} °")
    print(f"    EL          : {row['EL']:.4f} °")
    print()
    display(df_obs[["Epoch (COL)", "Epoch (UT)", "RA", "DEC", "AZ", "EL"]])

    # ── Plot RA – DEC ─────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(
        df_obs["RA"], df_obs["DEC"],
        "b-o", markersize=3, linewidth=1,
        label="Trayectoria Sol",
    )
    ax.plot(
        row["RA"], row["DEC"],
        "ro", markersize=10, zorder=5,
        label=(
            f"{t_col_str} COL\n"
            f"RA = {row['RA']:.4f}°\n"
            f"DEC = {row['DEC']:.4f}°"
        ),
    )
    ax.set_xlabel("RA (°)")
    ax.set_ylabel("DEC (°)")
    ax.set_title(f"RA – DEC del Sol  |  {t_col_str} COL  (±30 min, paso 1 min)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()

    # ── Plot AZ – EL ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(
        df_obs["AZ"], df_obs["EL"],
        "b-o", markersize=3, linewidth=1,
        label="Trayectoria Sol",
    )
    ax.plot(
        row["AZ"], row["EL"],
        "ro", markersize=10, zorder=5,
        label=(
            f"{t_col_str} COL\n"
            f"AZ = {row['AZ']:.4f}°\n"
            f"EL = {row['EL']:.4f}°"
        ),
    )
    ax.set_xlabel("AZ (°)")
    ax.set_ylabel("EL (°)")
    ax.set_title(f"AZ – EL del Sol  |  {t_col_str} COL  (±30 min, paso 1 min)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()